# 03 · Business questions

Real trading questions, the kind the Power BI report answers, using CTEs (`WITH`) and window functions. Each one is worth checking against a number on the report.

In [1]:
from _setup import q

## Check a number against the report

The Overview page says BY25 net sales were about £39.0M, up 4.5% on last year. If SQL agrees, both are reading the same warehouse correctly.

In [2]:
q("""
SELECT d.business_year,
       ROUND(SUM(f.net_sales_gbp) / 1e6, 2) AS net_sales_m
FROM fact_sales f
JOIN dim_date d ON f.date = d.full_date
GROUP BY d.business_year
ORDER BY d.business_year
""")

,business_year,net_sales_m
0,2023,37.15
1,2024,37.31
2,2025,39
3,2026,22.71


## This year against last year

`SUM(CASE WHEN ... THEN ... END)` puts two years side by side in one row, which is how you get a growth %. BY25 and BY24 are both complete years, so the comparison is fair.

In [3]:
q("""
WITH by_market AS (
    SELECT s.market_name,
           SUM(CASE WHEN d.business_year = 2025 THEN f.net_sales_gbp END) AS this_year,
           SUM(CASE WHEN d.business_year = 2024 THEN f.net_sales_gbp END) AS last_year
    FROM fact_sales f
    JOIN dim_date d  ON f.date = d.full_date
    JOIN dim_store s ON f.store_id = s.store_id
    WHERE d.business_year IN (2024, 2025)
    GROUP BY s.market_name
)
SELECT market_name,
       ROUND(this_year, 0) AS this_year,
       ROUND(last_year, 0) AS last_year,
       ROUND(100 * (this_year / last_year - 1), 1) AS growth_pct
FROM by_market
ORDER BY growth_pct DESC
""")

,market_name,this_year,last_year,growth_pct
0,Poland,"3,305,145","3,053,842",8.20
1,Latvia,"1,030,344","951,923",8.20
2,Slovakia,"889,022","827,897",7.40
3,Italy,"3,725,659","3,492,216",6.70
4,Czechia,"1,136,120","1,072,887",5.90
5,Ireland,"3,968,761","3,779,848",5
6,Lithuania,"770,192","735,104",4.80
7,United Kingdom,"14,751,294","14,139,423",4.30
8,Netherlands,"2,730,143","2,622,110",4.10
9,Germany,"5,602,809","5,547,707",1


## Ranking inside a group

A window function works across rows without collapsing them. `ROW_NUMBER() OVER (PARTITION BY major_product_group ORDER BY net_sales DESC)` numbers the product groups 1, 2, 3 within each major group.

`QUALIFY` filters on a window function. It works in DuckDB and Snowflake. In SQL Server you'd put the ranking in a CTE and filter it with `WHERE` instead.

In [4]:
q("""
WITH groups AS (
    SELECT p.major_product_group, p.product_group,
           SUM(f.net_sales_gbp) AS net_sales
    FROM fact_sales f
    JOIN dim_date d    ON f.date = d.full_date
    JOIN dim_product p ON f.sku = p.sku
    WHERE d.business_year = 2025
    GROUP BY p.major_product_group, p.product_group
)
SELECT major_product_group, product_group, ROUND(net_sales, 0) AS net_sales,
       ROW_NUMBER() OVER (PARTITION BY major_product_group ORDER BY net_sales DESC) AS rank_in_group
FROM groups
QUALIFY rank_in_group <= 3
ORDER BY major_product_group, rank_in_group
""")

,major_product_group,product_group,net_sales,rank_in_group
0,Accessories,Socks,"494,141",1
1,Accessories,Gloves,"413,181",2
2,Accessories,Accessory Sets,"136,152",3
3,Camping & Equipment,Sleeping Bags,"55,485",1
4,Camping & Equipment,Rucksacks,"23,597",2
...,...,...,...,...
16,Outerwear,Waterproof Insulated Jacket,"2,629,066",2
17,Outerwear,3 in 1,"2,096,125",3
18,Tops,Shirts,"1,181,415",1
19,Tops,Vests,"948,684",2


## Running totals: the Pareto

`SUM(...) OVER (ORDER BY ...)` gives a running total. Divide it by the grand total (another window, with nothing in the brackets) and you get the cumulative share behind the Style Pareto chart.

In [5]:
q("""
WITH styles AS (
    SELECT p.style_code, p.style_name, SUM(f.net_sales_gbp) AS net_sales
    FROM fact_sales f
    JOIN dim_date d    ON f.date = d.full_date
    JOIN dim_product p ON f.sku = p.sku
    WHERE d.business_year = 2025
    GROUP BY p.style_code, p.style_name
)
SELECT style_name,
       ROUND(net_sales, 0) AS net_sales,
       ROUND(100 * SUM(net_sales) OVER (ORDER BY net_sales DESC) / SUM(net_sales) OVER (), 1) AS cumulative_pct
FROM styles
ORDER BY net_sales DESC
LIMIT 10
""")

,style_name,net_sales,cumulative_pct
0,Denali Boots,"690,625",1.80
1,Helvellyn 3 in 1,"677,345",3.50
2,Ben Nevis Baffled/Quilted,"517,135",4.80
3,Chamonix Waterproof Insulated Jacket,"490,155",6.10
4,Cuillin Waterproof Insulated Jacket,"434,282",7.20
5,Kilimanjaro 3 in 1,"418,784",8.30
6,Torridon Waterproof Insulated Jacket,"413,748",9.30
7,Denali Waterproof Shell,"413,482",10.40
8,Annapurna Waterproof Shell,"389,060",11.40
9,Scafell Shoes,"376,051",12.40


And the headline: what share of styles make 80% of sales? The Products page says about 26%.

In [6]:
q("""
WITH styles AS (
    SELECT p.style_code, SUM(f.net_sales_gbp) AS net_sales
    FROM fact_sales f
    JOIN dim_date d    ON f.date = d.full_date
    JOIN dim_product p ON f.sku = p.sku
    WHERE d.business_year = 2025
    GROUP BY p.style_code
),
running AS (
    SELECT net_sales,
           SUM(net_sales) OVER (ORDER BY net_sales DESC) / SUM(net_sales) OVER () AS cumulative_share,
           ROW_NUMBER() OVER (ORDER BY net_sales DESC) AS style_rank,
           COUNT(*) OVER () AS styles
    FROM styles
)
SELECT MIN(style_rank) AS styles_to_reach_80pct,
       MAX(styles) AS total_styles,
       ROUND(100.0 * MIN(style_rank) / MAX(styles), 1) AS share_of_styles_pct
FROM running
WHERE cumulative_share >= 0.8
""")

,styles_to_reach_80pct,total_styles,share_of_styles_pct
0,235,907,25.90


## Challenges

**3.1** Which ten stores grew most in BY25 against BY24? Only include stores that traded in both years.

*Hint: the same `CASE WHEN` pattern as the market query, then `WHERE last_year > 0`.*

In [7]:
# your query here
# q("""
#     SELECT ...
# """)

**3.2** What was the best-selling style in each market in BY25?

*Hint: rank styles within each market with `PARTITION BY market_name`, then keep rank 1.*

In [8]:
# your query here
# q("""
#     SELECT ...
# """)

**3.3** Net sales by business period for BY25, with a running total. Use `business_period_label`, and order by `period_key` so the periods come out in order.

In [9]:
# your query here
# q("""
#     SELECT ...
# """)

**3.4** Reproduce the BY25 gross margin % on the Overview page (68.9%).

In [10]:
# your query here
# q("""
#     SELECT ...
# """)

---
## Answers

**3.1**

In [11]:
q("""
WITH by_store AS (
    SELECT s.store_name, s.market_name,
           SUM(CASE WHEN d.business_year = 2025 THEN f.net_sales_gbp END) AS this_year,
           SUM(CASE WHEN d.business_year = 2024 THEN f.net_sales_gbp END) AS last_year
    FROM fact_sales f
    JOIN dim_date d  ON f.date = d.full_date
    JOIN dim_store s ON f.store_id = s.store_id
    WHERE d.business_year IN (2024, 2025)
    GROUP BY s.store_name, s.market_name
)
SELECT store_name, market_name,
       ROUND(100 * (this_year / last_year - 1), 1) AS growth_pct
FROM by_store
WHERE last_year > 0
ORDER BY growth_pct DESC
LIMIT 10
""")

,store_name,market_name,growth_pct
0,Anchor Point Retail — Wroclaw,Poland,24.40
1,Solstice Outlets — Ostrava,Czechia,23.20
2,Greenway Garden Centre — Liepaja,Latvia,19.60
3,Areta Warsaw (2),Poland,18.70
4,Solstice Outlets — Warsaw,Poland,18.60
5,Anchor Point Retail — York (2),United Kingdom,18
6,Solstice Outlets — Rome (2),Italy,16.80
7,Millbrook Department Store — York,United Kingdom,16.70
8,Harbourside Retail Park — Sheffield,United Kingdom,16
9,Greenway Garden Centre — Daugavpils,Latvia,16


**3.2**

In [12]:
q("""
WITH style_sales AS (
    SELECT s.market_name, p.style_name, p.brand_name,
           SUM(f.net_sales_gbp) AS net_sales
    FROM fact_sales f
    JOIN dim_date d    ON f.date = d.full_date
    JOIN dim_store s   ON f.store_id = s.store_id
    JOIN dim_product p ON f.sku = p.sku
    WHERE d.business_year = 2025
    GROUP BY s.market_name, p.style_code, p.style_name, p.brand_name
)
SELECT market_name, style_name, brand_name, ROUND(net_sales, 0) AS net_sales
FROM style_sales
QUALIFY ROW_NUMBER() OVER (PARTITION BY market_name ORDER BY net_sales DESC) = 1
ORDER BY net_sales DESC
""")

,market_name,style_name,brand_name,net_sales
0,United Kingdom,Denali Boots,Areta,"320,172"
1,Germany,Helvellyn 3 in 1,Areta,"102,998"
2,Italy,Torridon Waterproof Insulated Jacket,Areta,"76,356"
3,Ireland,Cuillin Waterproof Insulated Jacket,Areta Pro,"71,404"
4,Poland,Helvellyn 3 in 1,Areta,"65,916"
5,Netherlands,Cairngorm Overtrousers,Kestrel Ridge,"52,521"
6,Lithuania,Denali Boots,Areta,"32,858"
7,Latvia,Helvellyn 3 in 1,Areta,"30,008"
8,Czechia,Eiger Fleece,Kestrel Ridge,"29,352"
9,France,Aconcagua Shorts,Areta,"27,277"


**3.3**

In [13]:
q("""
WITH periods AS (
    SELECT d.period_key, d.business_period_label,
           SUM(f.net_sales_gbp) AS net_sales
    FROM fact_sales f
    JOIN dim_date d ON f.date = d.full_date
    WHERE d.business_year = 2025
    GROUP BY d.period_key, d.business_period_label
)
SELECT business_period_label,
       ROUND(net_sales, 0) AS net_sales,
       ROUND(SUM(net_sales) OVER (ORDER BY period_key), 0) AS running_total
FROM periods
ORDER BY period_key
""")

,business_period_label,net_sales,running_total
0,BY25 P01,"2,537,519","2,537,519"
1,BY25 P02,"2,500,489","5,038,008"
2,BY25 P03,"3,199,364","8,237,372"
3,BY25 P04,"2,783,367","11,020,739"
4,BY25 P05,"2,777,143","13,797,882"
5,BY25 P06,"4,074,976","17,872,858"
6,BY25 P07,"3,450,195","21,323,053"
7,BY25 P08,"3,564,447","24,887,501"
8,BY25 P09,"4,561,718","29,449,219"
9,BY25 P10,"3,359,400","32,808,618"


**3.4**

In [14]:
q("""
SELECT ROUND(100 * SUM(f.net_sales_gbp - f.cost_gbp) / SUM(f.net_sales_gbp), 1) AS gross_margin_pct
FROM fact_sales f
JOIN dim_date d ON f.date = d.full_date
WHERE d.business_year = 2025
""")

,gross_margin_pct
0,68.90
